#download dataset

In [11]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("amananandrai/ag-news-classification-dataset")

print("Path to dataset files:", path)

100%|██████████| 11.4M/11.4M [00:00<00:00, 96.8MB/s]

Extracting files...


Path to dataset files: /root/.cache/kagglehub/datasets/amananandrai/ag-news-classification-dataset/versions/2


# List the files

In [12]:
import os

print(path)
print(os.listdir(path))

/root/.cache/kagglehub/datasets/amananandrai/ag-news-classification-dataset/versions/2
['test.csv', 'train.csv']


# Read the CSV files

In [13]:
import pandas as pd

train_df = pd.read_csv(f"{path}/train.csv")
test_df = pd.read_csv(f"{path}/test.csv")

# Check the dataset

In [14]:
train_df.head()

,Class Index,Title,Description
0,3,Wall St. Bears Claw Back Into the Black (Reuters),"Reuters - Short-sellers, Wall Street's dwindli..."
1,3,Carlyle Looks Toward Commercial Aerospace (Reu...,Reuters - Private investment firm Carlyle Grou...
2,3,Oil and Economy Cloud Stocks' Outlook (Reuters),Reuters - Soaring crude prices plus worries\ab...
3,3,Iraq Halts Oil Exports from Main Southern Pipe...,Reuters - Authorities have halted oil export\f...
4,3,"Oil prices soar to all-time record, posing new...","AFP - Tearaway world oil prices, toppling reco..."


In [15]:
train_df.info()
train_df.columns

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 120000 entries, 0 to 119999
Data columns (total 3 columns):
 #   Column       Non-Null Count   Dtype 
---  ------       --------------   ----- 
 0   Class Index  120000 non-null  int64 
 1   Title        120000 non-null  object
 2   Description  120000 non-null  object
dtypes: int64(1), object(2)
memory usage: 2.7+ MB


Index(['Class Index', 'Title', 'Description'], dtype='object')

# Combine Title and Description

In [16]:
train_df["text"] = train_df["Title"] + ". " + train_df["Description"]
test_df["text"] = test_df["Title"] + ". " + test_df["Description"]

# Keep only the columns needed for BERT

In [17]:
train_df = train_df[["text", "Class Index"]]
test_df = test_df[["text", "Class Index"]]

# Rename the label column

In [18]:
train_df.rename(columns={"Class Index": "label"}, inplace=True)
test_df.rename(columns={"Class Index": "label"}, inplace=True)

# Convert labels from 1–4 to 0–3

In [19]:
train_df["label"] = train_df["label"] - 1
test_df["label"] = test_df["label"] - 1

# Check the final dataset

In [20]:
train_df.head()

,text,label
0,Wall St. Bears Claw Back Into the Black (Reute...,2
1,Carlyle Looks Toward Commercial Aerospace (Reu...,2
2,Oil and Economy Cloud Stocks' Outlook (Reuters...,2
3,Iraq Halts Oil Exports from Main Southern Pipe...,2
4,"Oil prices soar to all-time record, posing new...",2


# Phase 2: Tokenization and Preprocessing

In [21]:
!pip install transformers datasets evaluate accelerate -q #Step 2.1: Install Required Libraries

In [22]:
#Step 2.2: Import Required Libraries
import pandas as pd
import numpy as np
import torch

from transformers import BertTokenizer
from datasets import Dataset

In [23]:
#Step 2.3: Convert Pandas DataFrames to Hugging Face Datasets
train_dataset = Dataset.from_pandas(train_df)
test_dataset = Dataset.from_pandas(test_df)

print(train_dataset)
print(test_dataset)

Dataset({
    features: ['text', 'label'],
    num_rows: 120000
})
Dataset({
    features: ['text', 'label'],
    num_rows: 7600
})


In [24]:
#Step 2.4: Load the BERT Tokenizer
tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

In [25]:
#Step 2.5: Define the Tokenization Function
def tokenize_function(example):
    return tokenizer(
        example["text"],
        truncation=True,
        padding="max_length",
        max_length=128
    )

In [26]:
#Step 2.6: Tokenize the Training and Test Datasets
tokenized_train = train_dataset.map(tokenize_function, batched=True)
tokenized_test = test_dataset.map(tokenize_function, batched=True)

Map:   0%|          | 0/120000 [00:00<?, ? examples/s]

Map:   0%|          | 0/7600 [00:00<?, ? examples/s]

In [27]:
#Step 2.7: Verify the Tokenized Dataset input_ids: numerical IDs for the tokens.
#token_type_ids: distinguishes sentence pairs (all zeros here since each example is a single text).
#attention_mask: tells BERT which tokens are real text (1) and which are padding (0).
print(tokenized_train)

Dataset({
    features: ['text', 'label', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 120000
})


# Phase 3: Fine-Tune BERT
Load the pre-trained BERT model.
Set up the evaluation metrics (Accuracy and F1-score).
Configure the training settings.
Train (fine-tune) the model.

In [1]:
#Step 3.1: Load the Pre-trained BERT Model
from transformers import BertForSequenceClassification

model = BertForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=4
)

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [3]:
import numpy as np

!pip install evaluate -q
import evaluate

accuracy = evaluate.load("accuracy")
f1 = evaluate.load("f1")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 2.8 MB/s eta 0:00:00


In [4]:
#Step 3.3: Create the Metric Function
def compute_metrics(eval_pred):
    logits, labels = eval_pred

    predictions = np.argmax(logits, axis=-1)

    acc = accuracy.compute(
        predictions=predictions,
        references=labels
    )

    f1_score = f1.compute(
        predictions=predictions,
        references=labels,
        average="weighted"
    )

    return {
        "accuracy": acc["accuracy"],
        "f1": f1_score["f1"]
    }

In [5]:
#Step 3.4: Set the Training Arguments
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=2,
    weight_decay=0.01,
    logging_dir="./logs",
    logging_steps=100
)

[transformers] `logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


In [28]:
# Step 3.5:create the trainer

from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_test,
    compute_metrics=compute_metrics,
)

In [ ]:
#Step 3.6: Train the Model
trainer.train()

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss


In [ ]:
Step 3.7: Evaluate the Model
results = trainer.evaluate()

print(results)

In [ ]:
Step 3.8: Save the Fine-Tuned Model
trainer.save_model("bert_news_classifier")
tokenizer.save_pretrained("bert_news_classifier")

# Phase 4: Model Deployment Using Gradio

In [ ]:
#Step 4.1: Install Gradio
!pip install -q gradio

In [ ]:
#Step 4.2: Import the Required Libraries
import torch import gradio as gr from transformers import BertTokenizer, BertForSequenceClassification

In [ ]:
#Step 4.3: Load the Saved Model
model = BertForSequenceClassification.from_pretrained("bert_news_classifier") tokenizer = BertTokenizer.from_pretrained("bert_news_classifier") model.eval()

In [ ]:
#Step 4.4: Define the News Categories
labels = { 0: "World", 1: "Sports", 2: "Business", 3: "Science/Technology" }

In [ ]:
#Step 4.5: Create the Prediction Function
def predict_news(text): inputs = tokenizer( text, return_tensors="pt", truncation=True, padding=True, max_length=128 ) with torch.no_grad(): outputs = model(**inputs) prediction = torch.argmax(outputs.logits, dim=1).item() return labels[prediction]

In [ ]:
#Step 4.6: Build the Gradio Interface
interface = gr.Interface( fn=predict_news, inputs=gr.Textbox( lines=2, placeholder="Enter a news headline..." ), outputs="text", title="News Topic Classifier Using BERT", description="Enter a news headline or short article to predict its category." ) interface.launch()